In [1]:
# =========================================================
# DCC-GARCH
# - Univariate GARCH(1,1) for each series
# - Standardized residuals
# - DCC(1,1) estimation
# - Outputs: dynamic correlation matrices
# =========================================================

# !pip install arch

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.linalg import det
from arch import arch_model

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "./spillover_data.csv"
OUT_DIR = "./output_dcc_garch"

DATE_COL = "Date"
VARS = ["dlog_SOLVPN", "dlog_COPPER", "dlog_DXY", "d_UST10Y", "dlog_VIX"]

MEAN_LAGS = 1
VOL_MODEL = "Garch"
P_GARCH = 1
Q_GARCH = 1
DIST = "normal"   # 'normal' or 't'

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[[DATE_COL] + VARS].dropna().reset_index(drop=True)

dates = df[DATE_COL].copy()
X = df[VARS].copy()

# arch package often prefers scaled returns
X_scaled = X * 100.0

T, N = X_scaled.shape
print("Data shape:", X_scaled.shape)

# -----------------------------
# Step 1: Univariate GARCH fit
# -----------------------------
std_resids = []
cond_vols = []
uni_summary = []

for col in VARS:
    am = arch_model(
        X_scaled[col],
        mean="AR",
        lags=MEAN_LAGS,
        vol=VOL_MODEL,
        p=P_GARCH,
        q=Q_GARCH,
        dist=DIST,
        rescale=False
    )
    res = am.fit(disp="off")

    sigma = res.conditional_volatility
    resid = res.resid
    z = resid / sigma

    std_resids.append(z.values)
    cond_vols.append(sigma.values)

    uni_summary.append({
        "Variable": col,
        "AIC": res.aic,
        "BIC": res.bic,
        "LogLik": res.loglikelihood
    })

Z = np.column_stack(std_resids)
H_uni = np.column_stack(cond_vols)

# trim NaN
mask = np.isfinite(Z).all(axis=1) & np.isfinite(H_uni).all(axis=1)
Z = Z[mask]
H_uni = H_uni[mask]
dates2 = dates.loc[mask].reset_index(drop=True)

T2 = len(dates2)
print("After GARCH valid rows:", T2)

# -----------------------------
# Step 2: DCC log-likelihood
# -----------------------------
S = np.cov(Z.T)

def dcc_loglik(params, Z, S):
    a, b = params

    # stationarity / positivity
    if a < 0 or b < 0 or a + b >= 0.999:
        return 1e10

    T, N = Z.shape
    Q_t = S.copy()
    ll = 0.0

    for t in range(T):
        z_t = Z[t].reshape(-1, 1)

        if t > 0:
            z_prev = Z[t - 1].reshape(-1, 1)
            Q_t = (1 - a - b) * S + a * (z_prev @ z_prev.T) + b * Q_t

        d = np.sqrt(np.diag(Q_t))
        d[d < 1e-12] = 1e-12
        D_inv = np.diag(1.0 / d)
        R_t = D_inv @ Q_t @ D_inv
        R_t = (R_t + R_t.T) / 2

        try:
            sign, logdet = np.linalg.slogdet(R_t)
            if sign <= 0:
                return 1e10
            invR = np.linalg.pinv(R_t)
            ll += logdet + float(z_t.T @ invR @ z_t)
        except Exception:
            return 1e10

    return 0.5 * ll

x0 = np.array([0.03, 0.95])
bounds = [(1e-6, 0.5), (1e-6, 0.999)]
cons = [{"type": "ineq", "fun": lambda x: 0.999 - x[0] - x[1]}]

opt = minimize(
    dcc_loglik,
    x0=x0,
    args=(Z, S),
    method="SLSQP",
    bounds=bounds,
    constraints=cons,
    options={"maxiter": 500, "disp": True}
)

a_hat, b_hat = opt.x
print("Estimated DCC params:", opt.x)

# -----------------------------
# Step 3: Reconstruct dynamic correlations
# -----------------------------
Q_t = S.copy()
R_list = []
H_list = []
pair_records = []

for t in range(T2):
    z_t = Z[t].reshape(-1, 1)

    if t > 0:
        z_prev = Z[t - 1].reshape(-1, 1)
        Q_t = (1 - a_hat - b_hat) * S + a_hat * (z_prev @ z_prev.T) + b_hat * Q_t

    d = np.sqrt(np.diag(Q_t))
    d[d < 1e-12] = 1e-12
    D_inv = np.diag(1.0 / d)

    R_t = D_inv @ Q_t @ D_inv
    R_t = (R_t + R_t.T) / 2
    R_list.append(R_t)

    D_vol = np.diag(H_uni[t])
    H_t = D_vol @ R_t @ D_vol
    H_list.append(H_t)

    for i in range(N):
        for j in range(N):
            pair_records.append({
                "Date": dates2.iloc[t],
                "Var1": VARS[i],
                "Var2": VARS[j],
                "DCC_Corr": R_t[i, j],
                "Covariance": H_t[i, j]
            })

pair_df = pd.DataFrame(pair_records)
uni_df = pd.DataFrame(uni_summary)

# -----------------------------
# Save
# -----------------------------
pair_df.to_csv(os.path.join(OUT_DIR, "dcc_dynamic_pairs.csv"), index=False)
uni_df.to_csv(os.path.join(OUT_DIR, "univariate_garch_summary.csv"), index=False)

avg_corr = (
    pair_df.groupby(["Var1", "Var2"])["DCC_Corr"]
    .mean()
    .unstack()
    .reindex(index=VARS, columns=VARS)
)
avg_corr.to_csv(os.path.join(OUT_DIR, "dcc_average_correlation_matrix.csv"))

meta = {
    "model": "DCC-GARCH(1,1)",
    "vars": VARS,
    "mean_lags": MEAN_LAGS,
    "vol_model": VOL_MODEL,
    "p_garch": P_GARCH,
    "q_garch": Q_GARCH,
    "dist": DIST,
    "dcc_alpha": float(a_hat),
    "dcc_beta": float(b_hat),
    "n_obs_used": int(T2)
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# -----------------------------
# Plot key pairs
# -----------------------------
def plot_pair(var1, var2, fname):
    tmp = pair_df[(pair_df["Var1"] == var1) & (pair_df["Var2"] == var2)].copy()
    tmp["Date"] = pd.to_datetime(tmp["Date"])
    plt.figure(figsize=(12, 5))
    plt.plot(tmp["Date"], tmp["DCC_Corr"])
    plt.title(f"DCC Correlation: {var1} vs {var2}")
    plt.xlabel("Date")
    plt.ylabel("Correlation")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=300)
    plt.show()

plot_pair("dlog_SOLVPN", "dlog_COPPER", "dcc_corr_SOLVPN_COPPER.png")
plot_pair("dlog_SOLVPN", "dlog_VIX", "dcc_corr_SOLVPN_VIX.png")

print("Saved to:", OUT_DIR)
print(pair_df.tail())

ModuleNotFoundError: No module named 'arch'